In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import os
ProjDIR = "/home/jw3514/Work/CellType_Psy/CellTypeBias_VIP/" # Change to your project directory
sys.path.insert(1, f'{ProjDIR}/src/')
sys.path.insert(1, '/home/jw3514/Work/UNIMED/src')
from CellType_PSY import *
from UNIMED import *
#import scanpy as sc
HGNC, ENSID2Entrez, GeneSymbol2Entrez, Entrez2Symbol = LoadGeneINFO()

try:
    os.chdir(f"{ProjDIR}/notebooks/")
    print(f"Current working directory: {os.getcwd()}")
except FileNotFoundError as e:
    print(f"Error: Could not change directory - {e}")
except Exception as e:
    print(f"Unexpected error: {e}")

In [ ]:
ClusterAnn = pd.read_csv("/home/jw3514/Work/CellType_Psy/AllenBrainCellAtlas/MouseCT_Cluster_Anno.csv", index_col="cluster_id_label")
ALL_Mouse_Class = sorted(ClusterAnn["class_id_label"].unique())

In [ ]:
MouseCT_SpecMat = pd.read_csv("/home/jw3514/Work/CellType_Psy/dat/Test.BiasMat/MouseCT.cluster.filtTPM.spec.percentile.csv", index_col=0)

In [ ]:
GeneWeightDIR = "../dat/GeneWeights/"
#Bias_Save_Dict = "../dat/Z2_v2_Bias_Apr18_2025/"
#Bias_Save_Dict = "../dat/ZMatch_Bias_Apr18_2025/"
Bias_Save_Dict = "../dat/MouseCT_Spec_Percentile_Bias_June05_2025/"
#Bias_Save_Dict = "../dat/Spec_Bias_Jun04_2025/"
if not os.path.exists(Bias_Save_Dict): # make dir if not exists
    os.makedirs(Bias_Save_Dict)

In [ ]:
HIQ_GW = Fil2Dict("{}/HIQ.top61.nopLI.LGD_Dmis_SameWeight.gw".format(GeneWeightDIR))
LIQ_GW = Fil2Dict("{}/LIQ.top61.nopLI.LGD_Dmis_SameWeight.gw".format(GeneWeightDIR))
print(len(HIQ_GW), len(LIQ_GW))

In [ ]:
ASD_HIQ_Bias = MouseCT_AvgZ_Weighted(MouseCT_SpecMat, HIQ_GW)
#HIQ_Z2_Bias = AdjustClusterMean(HIQ_Z2_Bias, HumanCT_Z2_HCT_ColMean)
ASD_HIQ_Bias = add_class(ASD_HIQ_Bias, ClusterAnn)
ASD_HIQ_Bias.to_csv("{}/ASD61.HIQ.csv".format(Bias_Save_Dict))

ASD_LIQ_Bias = MouseCT_AvgZ_Weighted(MouseCT_SpecMat, LIQ_GW)
#LIQ_Z2_Bias = AdjustClusterMean(LIQ_Z2_Bias, HumanCT_Z2_HCT_ColMean)
ASD_LIQ_Bias = add_class(ASD_LIQ_Bias, ClusterAnn)
ASD_LIQ_Bias.to_csv("{}/ASD61.LIQ.csv".format(Bias_Save_Dict))

In [ ]:
plot_pc1_boxplot_mouseCT(ASD_HIQ_Bias, ClusterAnn, ALL_Mouse_Class, "EFFECT")
plot_pc1_boxplot_mouseCT(ASD_LIQ_Bias, ClusterAnn, ALL_Mouse_Class, "EFFECT")

In [ ]:
X22q_GW = Fil2Dict("../dat/GeneWeights/X22q.gw.csv")
X22q_GW_Mouse = Fil2Dict("{}/X22q.mousemodel.gw.csv".format(GeneWeightDIR))

In [ ]:
X22q_Bias = MouseCT_AvgZ_Weighted(MouseCT_SpecMat, X22q_GW_Mouse)
X22q_Bias = add_class(X22q_Bias, ClusterAnn)
X22q_Bias.to_csv("{}/MouseCT.X22q.csv".format(Bias_Save_Dict))


In [ ]:
plot_pc1_boxplot_mouseCT(X22q_Bias, ClusterAnn, ALL_Mouse_Class, "EFFECT")
#plot_class_effects(mouseCT_bias_spec_pc_scores_df, ClusterAnn, ALL_Mouse_Class, "PC1")

In [ ]:
X22q_CGE = X22q_Bias[X22q_Bias["class_id_label"]=="06 CTX-CGE GABA"]
X22q_CGE_VIP_Pos = X22q_CGE[X22q_CGE.index.str.contains('Vip', case=False)]
X22q_CGE_VIP_Neg = X22q_CGE[~X22q_CGE.index.str.contains('Vip', case=False)]


In [ ]:
VIP_Pos_Index = X22q_CGE_VIP_Pos.index.values
VIP_Neg_Index = X22q_CGE_VIP_Neg.index.values
dat1 = MouseCT_SpecMat.loc[6899,VIP_Pos_Index]
dat2 = MouseCT_SpecMat.loc[6899,VIP_Neg_Index]
plt.hist(dat1, bins=100, color='blue', alpha=0.5, label='VIP+')
plt.hist(dat2, bins=100, color='red', alpha=0.5, label='VIP-')
plt.legend()
plt.show()


In [ ]:
VIP_Pos_Index = X22q_CGE_VIP_Pos.index.values
VIP_Neg_Index = X22q_CGE_VIP_Neg.index.values
dat1 = MouseCT_SpecMat.loc[6899,VIP_Pos_Index]
dat2 = MouseCT_SpecMat.loc[6899,VIP_Neg_Index]

# Get cell type indices where expression > 0.5
high_exp_vip_pos = VIP_Pos_Index[dat1 > 0.5]
high_exp_vip_neg = VIP_Neg_Index[dat2 > 0.5]

print("VIP+ cell types with expression > 0.5:")
print(high_exp_vip_pos)
print("\nVIP- cell types with expression > 0.5:")
print(high_exp_vip_neg)

# Plot histogram for visualization
plt.hist(dat1, bins=100, color='blue', alpha=0.5, label='VIP+')
plt.hist(dat2, bins=100, color='red', alpha=0.5, label='VIP-')
plt.axvline(x=0.5, color='black', linestyle='--', label='Threshold')
plt.legend()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

# Set style

# Create a figure
plt.figure(figsize=(10,6), dpi=100)

# Combine the data and create labels
vip_pos_data = X22q_CGE_VIP_Pos['EFFECT']
vip_neg_data = X22q_CGE_VIP_Neg['EFFECT']
data = pd.concat([vip_pos_data, vip_neg_data])
labels = ['VIP+'] * len(vip_pos_data) + ['VIP-'] * len(vip_neg_data)
df = pd.DataFrame({'Group': labels, 'EFFECT': data})

# Perform Mann-Whitney U test
statistic, pvalue = stats.mannwhitneyu(vip_pos_data, vip_neg_data, alternative='two-sided')

# Create boxplot with enhanced styling
sns.boxplot(x='Group', y='EFFECT', data=df, width=0.5, linewidth=2)
sns.swarmplot(x='Group', y='EFFECT', data=df, color='0.25', size=4, alpha=0.5)

# Customize the plot
plt.title('Comparison of EFFECT between VIP+ and VIP- Cell Types', 
          fontsize=14, pad=20, fontweight='bold')
plt.xlabel('Cell Type', fontsize=12, labelpad=10)
plt.ylabel('Effect Size', fontsize=12, labelpad=10)

# Add p-value annotation
plt.text(0.5, plt.ylim()[1], f'MWU p-value: {pvalue:.2e}',
         horizontalalignment='center', verticalalignment='bottom',
         fontsize=10, style='italic')

# Adjust layout and display
plt.tight_layout()
plt.show()

In [ ]:
#ASD_HIQ_Bias[ASD_HIQ_Bias.index.str.contains('vip', case=False)]

In [ ]:
SCZ_GW = Fil2Dict("{}/SCZ.top61.nopLI.LGD_Dmis_SameWeight.exclude_Mis2.gw".format(GeneWeightDIR))

In [ ]:
SCZ_Z2_Bias = MouseCT_AvgZ_Weighted(MouseCT_SpecMat, SCZ_GW)
#SCZ_Z2_Bias = AdjustClusterMean(SCZ_Z2_Bias, MouseCT_Z2_MCT_ColMean)
SCZ_Z2_Bias = add_class(SCZ_Z2_Bias, ClusterAnn)
SCZ_Z2_Bias.to_csv("{}/MCT.SCZ61.Z2.Spec.csv".format(Bias_Save_Dict))

In [ ]:
plot_pc1_boxplot_mouseCT(SCZ_Z2_Bias, ClusterAnn, ALL_Mouse_Class, "EFFECT")

In [ ]:
SCZ_GW_500 = Fil2Dict("{}/SCZ.top200.nopLI.LGD_Dmis_SameWeight.exclude_Mis2.gw".format(GeneWeightDIR))
print(len(SCZ_GW_500))
SCZ_Bias_500 = MouseCT_AvgZ_Weighted(MouseCT_SpecMat, SCZ_GW_500)
#SCZ_Z2_Bias = AdjustClusterMean(SCZ_Z2_Bias, MouseCT_Z2_MCT_ColMean)
SCZ_Bias_500 = add_class(SCZ_Bias_500, ClusterAnn)
#SuperClusterBias_BoxPlot(SCZ_Bias_500, "SCZ HCT")

In [ ]:
plot_pc1_boxplot_mouseCT(SCZ_Bias_500, ClusterAnn, ALL_Mouse_Class, "EFFECT")